# Baseline Flood Prediction Model — Simple LSTM (without decoder)
> Updated model to a simple LSTM for easier training. This still respects the sequence nature of the data but is easier to train given the limited data (5 stations) and short training period.

| | |
|---|---|
| **Author** | Clayton Sasaki |
| **Last modified** | January 13, 2026 |
| **Competition** | [CodaBench #10801](https://www.codabench.org/competitions/10801/) |

In [1]:
!git clone https://github.com/iharp-institute/iHARP-ML-Challenge-2.git

fatal: destination path 'iHARP-ML-Challenge-2' already exists and is not an empty directory.


In [2]:
# ============================================
# Install & Import Dependencies
# ============================================
# !pip install scipy pandas

import pandas as pd
import numpy as np
from scipy.io import loadmat
from datetime import datetime, timedelta

# ============================================
# Helper Function: MATLAB datenum → datetime
# ============================================
def matlab2datetime(matlab_datenum):
    return datetime.fromordinal(int(matlab_datenum)) \
           + timedelta(days=matlab_datenum % 1) \
           - timedelta(days=366)

# ============================================
# Load .mat Dataset
# ============================================
data = loadmat('/content/iHARP-ML-Challenge-2/NEUSTG_19502020_12stations.mat')

threshold_data = loadmat('/content/iHARP-ML-Challenge-2/Seed_Coastal_Stations_Thresholds.mat')

lat = data['lattg'].flatten()
lon = data['lontg'].flatten()
sea_level = data['sltg']
threshold = threshold_data['thminor_stnd']
station_names = [s[0] for s in data['sname'].flatten()]
time = data['t'].flatten()
time_dt = np.array([matlab2datetime(t) for t in time])

# ============================================
# Select Target Stations
# ============================================
SELECTED_STATIONS = [
    'Annapolis', 'Atlantic_City', 'Charleston', 'Washington', 'Wilmington'
]

selected_idx = [station_names.index(st) for st in SELECTED_STATIONS]
selected_names = [station_names[i] for i in selected_idx]
selected_lat = lat[selected_idx]
selected_lon = lon[selected_idx]
selected_threshold = threshold[selected_idx]
selected_sea_level = sea_level[:, selected_idx]  # time × selected_stations

# ============================================
# Build Preview DataFrame
# ============================================
df_preview = pd.DataFrame({
    'time': np.tile(time_dt[:5], len(selected_names)),
    'station_name': np.repeat(selected_names, 5),
    'latitude': np.repeat(selected_lat, 5),
    'longitude': np.repeat(selected_lon, 5),
    'sea_level': selected_sea_level[:5, :].T.flatten()
})

# ============================================
# Print Data Head
# ============================================
print(f"Number of stations: {len(selected_names)}")
print(f"Sea level shape (time x stations): {selected_sea_level.shape}")
df_preview.head()

Number of stations: 5
Sea level shape (time x stations): (622392, 5)


,time,station_name,latitude,longitude,sea_level
0,1950-01-01 00:00:00.000000,Annapolis,38.98328,-76.4816,1.341
1,1950-01-01 00:59:59.999997,Annapolis,38.98328,-76.4816,1.311
2,1950-01-01 02:00:00.000003,Annapolis,38.98328,-76.4816,1.280
3,1950-01-01 03:00:00.000000,Annapolis,38.98328,-76.4816,1.280
4,1950-01-01 03:59:59.999997,Annapolis,38.98328,-76.4816,1.341


In [3]:
# ============================================
# Convert Hourly → Daily per Station
# ============================================
# Convert time to pandas datetime
time_dt = pd.to_datetime(time_dt)

# Build hourly DataFrame for selected stations
df_hourly = pd.DataFrame({
    'time': np.tile(time_dt, len(selected_names)),
    'station_name': np.repeat(selected_names, len(time_dt)),
    'latitude': np.repeat(selected_lat, len(time_dt)),
    'longitude': np.repeat(selected_lon, len(time_dt)),
    'sea_level': selected_sea_level.flatten(),
    'flood_threshold': np.repeat(selected_threshold, len(time_dt)),
})

# ============================================
# Compute Flood Threshold per Station
# ============================================
#threshold_df = df_hourly.groupby('station_name')['sea_level'].agg(['mean','std']).reset_index()
#threshold_df['flood_threshold'] = threshold_df['mean'] + 1.5 * threshold_df['std']


#df_hourly = df_hourly.merge(threshold_df[['station_name','threshold']], on='station_name', how='left')

# ============================================
# Daily Aggregation + Flood Flag
# ============================================
df_daily = df_hourly.groupby(['station_name', pd.Grouper(key='time', freq='D')]).agg({
    'sea_level': 'mean',
    'latitude': 'first',
    'longitude': 'first',
    'flood_threshold': 'first'
}).reset_index()

# Flood flag: 1 if any hourly value exceeded threshold that day
hourly_max = df_hourly.groupby(['station_name', pd.Grouper(key='time', freq='D')])['sea_level'].max().reset_index()
df_daily = df_daily.merge(hourly_max, on=['station_name','time'], suffixes=('','_max'))
df_daily['flood'] = (df_daily['sea_level_max'] > df_daily['flood_threshold']).astype(int)

# ============================================
# Feature Engineering (3d & 7d means)
# ============================================
df_daily['sea_level_3d_mean'] = df_daily.groupby('station_name')['sea_level'].transform(
    lambda x: x.rolling(3, min_periods=1).mean())
df_daily['sea_level_7d_mean'] = df_daily.groupby('station_name')['sea_level'].transform(
    lambda x: x.rolling(7, min_periods=1).mean())

# Preview
df_daily.head()

,station_name,time,sea_level,latitude,longitude,flood_threshold,sea_level_max,flood,sea_level_3d_mean,sea_level_7d_mean
0,Annapolis,1950-01-01,1.471958,38.98328,-76.4816,2.104,2.067,0,1.471958,1.471958
1,Annapolis,1950-01-02,1.455417,38.98328,-76.4816,2.104,2.505,1,1.463687,1.463687
2,Annapolis,1950-01-03,1.841542,38.98328,-76.4816,2.104,2.536,1,1.589639,1.589639
3,Annapolis,1950-01-04,1.396750,38.98328,-76.4816,2.104,1.737,0,1.564569,1.541417
4,Annapolis,1950-01-05,1.704333,38.98328,-76.4816,2.104,2.292,1,1.647542,1.574000


In [4]:
# ============================================
# Build 7-day → 14-day Training Windows
# ============================================
FEATURES = ['sea_level', 'sea_level_3d_mean', 'sea_level_7d_mean']
HIST_DAYS = 7
FUTURE_DAYS = 14

X_train, y_train = [], []

for stn, grp in df_daily.groupby('station_name'):
    grp = grp.sort_values('time').reset_index(drop=True)
    for i in range(len(grp) - HIST_DAYS - FUTURE_DAYS):
        hist = grp.loc[i:i+HIST_DAYS-1, FEATURES].values.flatten()
        future = grp.loc[i+HIST_DAYS:i+HIST_DAYS+FUTURE_DAYS-1, 'flood'].values
        X_train.append(hist)
        y_train.append(future)

X_train = np.array(X_train)
y_train = np.array(y_train)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

X_train shape: (129560, 21)
y_train shape: (129560, 14)


In [5]:
# ============================================
# Select Historical Window (Manual / Random)
# ============================================

# --- Option 1: RANDOM window ---
# np.random.seed(42)
# date_range = pd.date_range(start='1950-01-01', end='2020-12-15')
# hist_start = np.random.choice(date_range)
# hist_end = hist_start + pd.Timedelta(days=6)

# --- Option 2: MANUAL window ---
hist_start = pd.to_datetime('2013-07-21')
hist_end   = pd.to_datetime('2013-07-27')

# Forecast period
test_start = hist_end + pd.Timedelta(days=1)
test_end   = test_start + pd.Timedelta(days=13)

print(f"Historical window: {hist_start.date()} → {hist_end.date()}")
print(f"Forecast window:   {test_start.date()} → {test_end.date()}")

# ============================================
# Build X_test for Selected Window
# ============================================
FEATURES = ['sea_level', 'sea_level_3d_mean', 'sea_level_7d_mean']
X_test = []

for stn, grp in df_daily.groupby('station_name'):
    mask = (grp['time'] >= hist_start) & (grp['time'] <= hist_end)
    hist_block = grp.loc[mask, FEATURES].values.flatten()
    if len(hist_block) == 7 * len(FEATURES):   # ensure full 7-day block
        X_test.append(hist_block)

X_test = np.array(X_test)
print(f"X_test shape: {X_test.shape}  (stations × {7*len(FEATURES)} features)")

Historical window: 2013-07-21 → 2013-07-27
Forecast window:   2013-07-28 → 2013-08-10
X_test shape: (5, 21)  (stations × 21 features)


In [9]:
"""
Simplified LSTM for Coastal Flood Prediction
=============================================
Model section. Data preparation (reshape, normalize, Dataset, DataLoader) is above.

We use a single LSTM that reads the history and outputs all
14 predictions at once from the final hidden state.

X_train : (N_samples, 21)        — 7 days × 3 features, flattened
y_train : (N_samples, 14)        — 14-day binary flood labels
X_test  : (5, 21)                — one 7-day window per station
y_true  : (5, 14)                — ground truth for evaluation
"""

# ============================================
# Imports
# ============================================
# !pip install torch --quiet

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, matthews_corrcoef

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


# ============================================
# Step 1: Reshape Flat Vectors → Sequences
# ============================================
# The data prep code flattened 7 days × 3 features into a (21,) vector.
# The LSTM needs a proper time axis: (batch, time_steps, features).

HIST_DAYS   = 7
FUTURE_DAYS = 14
N_FEATURES  = 3   # sea_level, sea_level_3d_mean, sea_level_7d_mean

# (N_samples, 21) → (N_samples, 7, 3)
X_train_seq = X_train.reshape(-1, HIST_DAYS, N_FEATURES).astype(np.float32)
X_test_seq  = X_test.reshape(-1, HIST_DAYS, N_FEATURES).astype(np.float32)
y_train_f   = y_train.astype(np.float32)   # (N_samples, 14)


# ============================================
# Step 2: Normalize Inputs
# ============================================
# LSTMs are sensitive to input scale. We normalize each feature
# using training-set statistics only (no data leakage from test).

mean = X_train_seq.mean(axis=(0, 1), keepdims=True)   # (1, 1, 3)
std  = X_train_seq.std(axis=(0, 1), keepdims=True) + 1e-8

X_train_seq = (X_train_seq - mean) / std
X_test_seq  = (X_test_seq  - mean) / std


# ============================================
# Step 3: PyTorch Dataset & DataLoader
# ============================================

class FloodDataset(Dataset):
    """Wraps numpy arrays into a PyTorch-compatible dataset."""
    def __init__(self, X, y):
        self.X = torch.tensor(X)   # (N, 7, 3)
        self.y = torch.tensor(y)   # (N, 14)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = FloodDataset(X_train_seq, y_train_f)
train_loader  = DataLoader(train_dataset, batch_size=256, shuffle=True)


# ============================================
# Step 4: Simple LSTM Architecture
# ============================================

class SimpleLSTM(nn.Module):
    """
    A straightforward LSTM flood predictor.

    Instead of a separate encoder and decoder passing messages
    back and forth, this model does two simple things:

      1. Read the 7-day history with an LSTM
      2. Take the final hidden state and output all 14
         flood probabilities at once via a single linear layer

    Simple model as it is only learning from 5 stations.
    (No teacher forcing or autoregressive decoding)

    Input shape:  (batch, 7, 3)   — 7 days, 3 features
    Output shape: (batch, 14)     — 14 flood probabilities
    """
    def __init__(self, input_dim, hidden_dim, num_layers, dropout, future_days):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,     # 3 features per day
            hidden_size=hidden_dim,   # size of the memory vector
            num_layers=num_layers,    # stacked LSTM layers
            batch_first=True,         # expects (batch, seq, features)
            dropout=dropout if num_layers > 1 else 0.0
        )

        # One linear layer that maps the final memory
        # directly to 14 flood predictions
        self.output_layer = nn.Linear(hidden_dim, future_days)

    def forward(self, x):
        # x: (batch, 7, 3)

        # Run the LSTM over the 7-day sequence.
        # We only care about the final hidden state — the
        # LSTM's summary of everything it read.
        _, (hidden, _) = self.lstm(x)

        # hidden: (num_layers, batch, hidden_dim)
        # Take only the top layer's hidden state
        top_layer_hidden = hidden[-1]   # (batch, hidden_dim)

        # Map the memory to 14 predictions in one step.
        # We use BCEWithLogitsLoss later which applies
        # sigmoid internally (more numerically stable).
        out = self.output_layer(top_layer_hidden)   # (batch, 14)

        return out


# ============================================
# Step 5: Instantiate the Model
# ============================================

model = SimpleLSTM(
    input_dim=N_FEATURES,    # 3
    hidden_dim=64,           # Small memory vector: fast to train, less likely to overfit on only 5 stations worth of data
    num_layers=2,
    dropout=0.3,
    future_days=FUTURE_DAYS  # 14
).to(DEVICE)

print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")


# ============================================
# Step 6: Loss and Optimizer
# ============================================

# ~40% flood rate, nearly balanced dataset.
# Standard BCE works fine
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Reduce learning rate if loss plateaus for 10 epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=10, factor=0.5
)


# ============================================
# Step 7: Training Loop
# ============================================

EPOCHS = 150   # Giving model time to train

print(f"\nTraining on {DEVICE} for {EPOCHS} epochs...\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()


        preds = model(X_batch) # (batch, 14) raw logits

        loss = criterion(preds, y_batch)
        loss.backward()

        # Gradient clipping, good practice for LSTMs
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    scheduler.step(avg_loss)

    if epoch % 10 == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {avg_loss:.4f} | LR: {lr:.2e}")


# ============================================
# Step 8: Inference
# ============================================

model.eval()
with torch.no_grad():
    X_test_t = torch.tensor(X_test_seq).to(DEVICE)   # (5, 7, 3)

    raw_preds  = model(X_test_t)                          # (5, 14) logits
    y_prob     = torch.sigmoid(raw_preds).cpu().numpy()   # (5, 14) probabilities
    y_pred_bin = (y_prob > 0.5).astype(int)               # (5, 14) binary

# ============================================
# Step 9: Collect Ground Truth
# ============================================
y_true = []
for stn, grp in df_daily.groupby('station_name'):
    mask = (grp['time'] >= test_start) & (grp['time'] <= test_end)
    vals = grp.loc[mask, 'flood'].values
    if len(vals) == 14:
        y_true.append(vals)
y_true = np.array(y_true)

# ============================================
# Step 10: Evaluation
# ============================================

y_true_flat = y_true.flatten()
y_pred_flat = y_pred_bin.flatten()

tn, fp, fn, tp = confusion_matrix(y_true_flat, y_pred_flat).ravel()
acc = accuracy_score(y_true_flat, y_pred_flat)
f1  = f1_score(y_true_flat, y_pred_flat, zero_division=0)
mcc = matthews_corrcoef(y_true_flat, y_pred_flat)

print("\n=== Confusion Matrix ===")
print(f"TP: {tp} | FP: {fp} | TN: {tn} | FN: {fn}")
print("\n=== Metrics ===")
print(f"Accuracy : {acc:.3f}")
print(f"F1 Score : {f1:.3f}")
print(f"MCC      : {mcc:.3f}")

print("\n=== Per-Station Flood Probability (day-by-day) ===")
SELECTED_STATIONS = ['Annapolis', 'Atlantic_City', 'Charleston', 'Washington', 'Wilmington']
for i, stn in enumerate(SELECTED_STATIONS):
    probs_str = " ".join([f"{p:.2f}" for p in y_prob[i]])
    print(f"{stn:<16}: {probs_str}")


# ============================================
# Step 11: Health Check — Is the Data Correct?
# ============================================
# If after epoch 30 loss is still above 0.65 and barely
# changing, something upstream is wrong. This to verifies
# the training data looks reasonable.

print("\n=== Data Health Check ===")
print(f"X_train_seq shape : {X_train_seq.shape}")
print(f"y_train shape     : {y_train_f.shape}")
print(f"X_train_seq mean  : {X_train_seq.mean():.4f}  (should be near 0.0)")
print(f"X_train_seq std   : {X_train_seq.std():.4f}   (should be near 1.0)")
print(f"Flood rate        : {y_train_f.mean():.3f}    (fraction of flood days)")

Using device: cuda
SimpleLSTM(
  (lstm): LSTM(3, 64, num_layers=2, batch_first=True, dropout=0.3)
  (output_layer): Linear(in_features=64, out_features=14, bias=True)
)
Trainable parameters: 51,854

Training on cuda for 150 epochs...

Epoch  10/150 | Loss: 0.5439 | LR: 1.00e-03
Epoch  20/150 | Loss: 0.5385 | LR: 1.00e-03
Epoch  30/150 | Loss: 0.5345 | LR: 1.00e-03
Epoch  40/150 | Loss: 0.5298 | LR: 1.00e-03
Epoch  50/150 | Loss: 0.5252 | LR: 1.00e-03
Epoch  60/150 | Loss: 0.5213 | LR: 1.00e-03
Epoch  70/150 | Loss: 0.5160 | LR: 1.00e-03
Epoch  80/150 | Loss: 0.5120 | LR: 1.00e-03
Epoch  90/150 | Loss: 0.5081 | LR: 1.00e-03
Epoch 100/150 | Loss: 0.5051 | LR: 1.00e-03
Epoch 110/150 | Loss: 0.5016 | LR: 1.00e-03
Epoch 120/150 | Loss: 0.4976 | LR: 1.00e-03
Epoch 130/150 | Loss: 0.4949 | LR: 1.00e-03
Epoch 140/150 | Loss: 0.4925 | LR: 1.00e-03
Epoch 150/150 | Loss: 0.4894 | LR: 1.00e-03

=== Confusion Matrix ===
TP: 21 | FP: 3 | TN: 34 | FN: 12

=== Metrics ===
Accuracy : 0.786
F1 Score : 0